# Importing all functions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# creating incremental flag

In [0]:
dbutils.widgets.text("incremental_flag",'0')

In [0]:
incremental_flag = dbutils.widgets.get('incremental_flag')

# connection to ADLS

In [0]:
spark.conf.set("fs.azure.account.key.saleessa.dfs.core.windows.net","access_key")

# defining paths

In [0]:
silver_path = "abfss://silver@saleessa.dfs.core.windows.net/"
gold_path = "abfss://gold@saleessa.dfs.core.windows.net/"

# selecting product data

In [0]:
df_src = spark.sql(
    ''' select distinct(Product_Category) as Product_Category from 
        parquet.`abfss://silver@saleessa.dfs.core.windows.net/`
    
    '''
)

In [0]:
df_src.display()

Product_Category
SPORTS
DIY
AUTOMOTIVE
GROCERIES
JEWELRY
APPLIANCES
BOOKS
OFFICE SUPPLIES
TOYS
BABY PRODUCTS


# creating sink dataset

In [0]:
from delta.tables import DeltaTable

path =  "abfss://gold@saleessa.dfs.core.windows.net/dim_product"

if DeltaTable.isDeltaTable(spark,path):
    df_sink = spark.read.format('delta').load(path).select("product_key","Product_Category")
else:
    df_sink = spark.createDataFrame([], "product_key int,Product_Category string")
 


In [0]:
df = df_src.join(df_sink,on='Product_Category',how='left').select(df_src.Product_Category,df_sink.product_key)


Product_Category,product_key
SPORTS,null
DIY,null
AUTOMOTIVE,null
GROCERIES,null
JEWELRY,null
APPLIANCES,null
BOOKS,null
OFFICE SUPPLIES,null
TOYS,null
BABY PRODUCTS,null


# old and new records

In [0]:
df_old = df.filter(df.product_key.isNotNull())
df_new = df.filter(df.product_key.isNull())

In [0]:
if incremental_flag=='0':
    max_value = 1
else:
    max_value = df_old.select(max(df_old.product_key)).collect()[0][0]

max_value

1

# adding serogate keys to new data

In [0]:
df_new = df_new.withColumn('product_key',max_value + monotonically_increasing_id())

In [0]:
union_df = df_old.union(df_new)

In [0]:
path = "abfss://gold@saleessa.dfs.core.windows.net/dim_product"
file_name = 'dim_product'

if DeltaTable.isDeltaTable(spark,path):
    deltatable = DeltaTable.forPath(spark,path)
    deltatable.alias('target').merge(union_df.alias('source'),'target.product_key = source.product_key').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print('file upserted')

else:
    union_df.write.format('delta').mode('overwrite').option('mergeSchema','true').save(path)
    print('file created')

file created
